# Predicitive Analysis- Model Training for Classification

In this notebook, we will be training a Random Forest Classifier model for machine failure prediction. We begin my determining a class-weight ratio to handle the class imbalance in our dataset, then train a Random Forest model on the dataset. After training, we evaluate the model's perfomance on the validation and final datasets and export the model for deployment.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PATH = 'drive/MyDrive/predictive-analysis-data/classification/'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

### Loading the dataset

In [4]:
# loading the dataframes
train_df = pd.read_csv(f"{PATH}train.csv")
val_df = pd.read_csv(f"{PATH}val.csv")
test_df = pd.read_csv(f"{PATH}test.csv")

In [6]:
# excluding metadata & failure target variable from dataset
exclude_cols = ['machineID', 'datetime', 'target_failure_24h']

feature_cols = [col for col in train_df.columns if col not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df['target_failure_24h']

X_val = val_df[feature_cols]
y_val = val_df['target_failure_24h']

X_test = test_df[feature_cols]
y_test = test_df['target_failure_24h']

In [7]:
# storing metadata for later analysis
train_meta = train_df[['machineID', 'datetime']].copy()
val_meta = val_df[['machineID', 'datetime']].copy()
test_meta = test_df[['machineID', 'datetime']].copy()

### Peek into our data

In [8]:
print(f"\nDataset Summary:")
print(f"Train samples: {len(X_train):,}")
print(f"Val samples: {len(X_val):,}")
print(f"Test samples: {len(X_test):,}")
print(f"Total features: {X_train.shape[1]}")

print(f"\nTarget Distribution:")
train_failure_rate = y_train.mean()
val_failure_rate = y_val.mean()
test_failure_rate = y_test.mean()

print(f"Train: {y_train.sum():,} failures ({train_failure_rate*100:.2f}%)")
print(f"Val: {y_val.sum():,} failures ({val_failure_rate*100:.2f}%)")
print(f"Test: {y_test.sum():,} failures ({test_failure_rate*100:.2f}%)")



Dataset Summary:
Train samples: 729,000
Val samples: 72,000
Test samples: 75,100
Total features: 59

Target Distribution:
Train: 14,502 failures (1.99%)
Val: 1,374 failures (1.91%)
Test: 1,308 failures (1.74%)



Our dataset shows a severe class imbalance, where the minority class (failure) constitutes only 1.99% of the training data. This disparity can our cause models to ignore the rare failure cases. To address this imbalance, we apply class weighting by calculating the inverse ratio of the failure rate: 1 / 0.0199 = ~ 50. We will then use the resulting class weight dict: **{0: 1, 1: 50}** to ensure the model places 50 times the importance on correctly predicting failure instances.

In [9]:
class_weight_ratio = int(1 / train_failure_rate)
print(f"\nClass weight ratio: {{0: 1, 1: {class_weight_ratio}}}")


Class weight ratio: {0: 1, 1: 50}


### Training Random Forest Model

We began by training a baseline Random Forest model with some initial default parameters. After observing the high performance of this configuration on the validation set, we settled on these settings as our final model. <br>Specifically, the model is configured with 100 decision trees and utilizes the weighted class weight ({0: 1, 1: 50}) to effectively manage the severe class imbalance. We also use a fixed seed (42) to ensure reproducibility.

In [10]:
# assigning class weight
class_weight = {0: 1, 1: class_weight_ratio}

# training the model
base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight=class_weight,
    random_state=42,
    n_jobs=-1
)

base_model.fit(X_train, y_train)


RandomForestClassifier(class_weight={0: 1, 1: 50}, n_jobs=-1, random_state=42)

#### Evaluating the model on validation set

After training, we want to see how well the model performs on new unseen data. For now, we evaluate the model's performance on our validation dataset by caclulating these metrics: precision, recall, f1 score. These performance metrics helps us identify the rates of True Positives among predicted positives and actual positives, better than just Accuracy rate which can often be misleading, specially when there is class imbalance.

In [11]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    precision_recall_curve, average_precision_score,
    roc_auc_score, roc_curve,
)

In [12]:
# get predictions on val dataset
y_val_prediction = base_model.predict(X_val)

In [14]:
val_precision = precision_score(y_val, y_val_prediction)
val_recall = recall_score(y_val, y_val_prediction)
val_f1 = f1_score(y_val, y_val_prediction)

print(f"\nPrecision: {val_precision:.4f}")
print(f"Recall:    {val_recall:.4f}")
print(f"F1-Score:  {val_f1:.4f}")


Precision: 0.9510
Recall:    0.9898
F1-Score:  0.9700


From the performance metrics, we can see that our model performs exceptionally well on its prediction with high confidence. <br>
The Recall score of $0.9898$ shows the model successfully identifies almost $99\%$ of all actual failures, while the Precision of $0.9510$ means that $95\%$ of its failure predictions are correct, demonstrating that the class weighting strategy effectively worked on our model training, making it both sensitive and reliable in detecting rare failure events.

### Feature Importance Analysis

To understand which features have the highest influence on our model's predicition (and on a machine's failure), we are performing a feature importance analysis on the trained Random Forest model. This maps and sorts the most important/influential features for the model's decision-making process.

In [18]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': base_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))


10 Most Important Features:
                     feature  importance
       error_count_last_24_h    0.267347
      hours_since_last_error    0.201400
hours_since_last_maintenance    0.186495
     rotate_rolling_24h_mean    0.053137
       volt_rolling_24h_mean    0.039862
     days_since_last_failure    0.026062
  vibration_rolling_24h_mean    0.024746
      rotate_rolling_6h_mean    0.024560
        volt_rolling_6h_mean    0.023690
   pressure_rolling_24h_mean    0.021171


From the results, we can see that the model's decision to predict a failure is mainly affected by the recent frequency of errors. Specifically, the error count over the last 24 hours (26.73%) and the time elapsed since the last error (20.14%) are the two most important features. Together with hours since last maintenance (18.65%), these three features account for over $65\%$ of the model's predictive power, highlighting that recent history and maintenance context are the strongest indicators of failure.

### Threshold Validation

In imbalanced classification problems, the default probability threshold of 0.5 often performs poorly. As a necessary and a sanity check, we are checking our model's optimal threshold for maximum F1 score (balanced precision and recall). If the optimal threshold for our model is not 0.5, we set the new threshold for better performance.

In [21]:
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, y_val_prediction_prob)

# calculate F1 scores for each threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# find optimal threshold (maximizing F1)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds_pr[optimal_idx]

In [23]:
print(f"Default threshold: 0.5")
print(f"Optimal threshold: {optimal_threshold}")

  Default threshold: 0.5
  Optimal threshold: 0.5


The analysis confirms that the optimal threshold for maximizing the F1-Score is precisely **0.5** and shows that the model is well-calibrated and balanced.
<br>  We will proceed with the 0.5 default threshold for the final evaluation on the test set.

### Final Evaluation on test set

In [31]:
# get predictions on test dataset
y_test_prediction = base_model.predict(X_test)

In [32]:
# calculate all metrics
test_precision = precision_score(y_test, y_test_prediction)
test_recall = recall_score(y_test, y_test_prediction)
test_f1 = f1_score(y_test, y_test_prediction)

print(f"\nFinal test set perfomance metrics:")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1-Score: {test_f1:.4f}")


Final test set perfomance metrics:
Precision: 0.9770
Recall: 0.8784
F1-Score: 0.9251


The final perfomance metrics on the test set confirms the Random Forest model's strong performance and reliability in predicting machine failures.<br>
The model achieved a Precision of 0.9770, indicating that when it predicts a failure, it's correct over **97%** of the time. This demonstrates its reliability and minimal false alarms. The Recall of 0.8784 shows that the model successfully identified nearly **88%** of all actual failures in the test set, proving the model is highly sensitive to true failure events.

### Exporting trained model

In [35]:
import os
import pickle
os.makedirs(f"{PATH}models/", exist_ok=True)

In [36]:
# save the trained model
with open(f"{PATH}models/random_forest_final.pkl", 'wb') as f:
    pickle.dump(base_model, f)
print("Saved random_forest_final.pkl")

Saved random_forest_final.pkl
